In [16]:
import psycopg2

conn = psycopg2.connect("dbname=ecommerce_sales user=postgres password=lucifer54@")
cursor = conn.cursor()

In [17]:
import pandas as pd


In [18]:
data = pd.read_csv('../data/sales_data.csv')
data.head()

,OrderDate,ProductName,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53
2,2022-05-11,Tablet,Electronics,South,5,5865,502.73
3,2024-03-16,Mouse,Accessories,South,2,786,202.87
4,2022-09-10,Mouse,Accessories,West,1,509,103.28


Create tables


In [19]:
def create_table_regions():

  query = 'CREATE TABLE IF NOT EXISTS REGIONS(region_id int PRIMARY KEY, region_name varchar(25));'
  cursor.execute(query)
  conn.commit()
  print("Created table categories!")

create_table_regions()

Created table categories!


In [20]:
def create_table_categories():

  query = 'CREATE TABLE IF NOT EXISTS CATEGORIES(category_id int PRIMARY KEY, category_name varchar(25));'
  cursor.execute(query)
  conn.commit()
  print("Created table categories!")

create_table_categories()

Created table categories!


In [21]:
def create_table_products():

  query = 'CREATE TABLE IF NOT EXISTS PRODUCTS(product_id int PRIMARY KEY, product_name varchar(25), category_id int, FOREIGN KEY (category_id) REFERENCES CATEGORIES(category_id));'
  cursor.execute(query)
  conn.commit()
  print("Created table Products!")

create_table_products()

Created table Products!


In [22]:
product_names = list(set(data['ProductName']))
categories = set(data['Category'])

hash = {1: ['Printer'],2: ['Mouse','Keyboard'],3: ['Camera', 'Tablet', 'Headphones', 'Laptop', 'Monitor','Smartphone', 'Smartwatch']}


print(product_names)
print(categories)
products = []
for i in range(len(product_names)):
  cat = ''
  for key,val in hash.items():
    if product_names[i] in val:
      cat = key
      break
  pdt = (i+1,product_names[i],cat)
  products.append(pdt)
print(products)



['Headphones', 'Printer', 'Mouse', 'Smartwatch', 'Keyboard', 'Smartphone', 'Camera', 'Monitor', 'Tablet', 'Laptop']
{'Accessories', 'Office', 'Electronics'}
[(1, 'Headphones', 3), (2, 'Printer', 1), (3, 'Mouse', 2), (4, 'Smartwatch', 3), (5, 'Keyboard', 2), (6, 'Smartphone', 3), (7, 'Camera', 3), (8, 'Monitor', 3), (9, 'Tablet', 3), (10, 'Laptop', 3)]


In [23]:
def create_table_sales():

  query = 'CREATE TABLE IF NOT EXISTS SALES(sales_id SERIAL PRIMARY KEY, p_id int, r_id int,order_date DATE, quantity int, sales int,profit DOUBLE PRECISION, FOREIGN KEY(p_id) REFERENCES PRODUCTS(product_id), FOREIGN KEY(r_id) REFERENCES REGIONS(region_id));'
  cursor.execute(query)
  conn.commit()
  print("Created table Sales!")

create_table_sales()

Created table Sales!


Sales -> p_id, r_id, order_date, quantity, sales, profit

In [24]:
from keybert import KeyBERT

In [64]:
bert_model = KeyBERT('distilbert-base-nli-mean-tokens')


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2880.25it/s]


In [27]:
from sentence_transformers import SentenceTransformer


In [50]:
regions = ['North', 'South', 'East', 'West']
embed_target = product_names + list(categories) + regions
print(embed_target)

['Headphones', 'Printer', 'Mouse', 'Smartwatch', 'Keyboard', 'Smartphone', 'Camera', 'Monitor', 'Tablet', 'Laptop', 'Accessories', 'Office', 'Electronics', 'North', 'South', 'East', 'West']


In [51]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5721.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [53]:
embeddings = model.encode(embed_target)
print(embeddings.shape)

(17, 384)


In [42]:
from sklearn.metrics.pairwise import cosine_similarity

In [59]:
sims = cosine_similarity([userQuery],embeddings)
print(sims)

[[0.13981135 0.21303949 0.16181959 0.08976768 0.2008394  0.10938263
  0.23385352 0.14727092 0.14361614 0.2486044  0.11928572 0.23802143
  0.21886453 0.67754126 1.         0.6193468  0.63196385]]


In [123]:
userQuery = "What category does keyboard come under?"

In [124]:
userQuery = userQuery.split()

for i in range(len(userQuery)):
  enc_query = model.encode(userQuery[i])
  sims = list(cosine_similarity([enc_query],embeddings))
  trust = max(sims[0])
  if trust > 0.6:
    print(trust)
    userQuery[i] = embed_target[list(sims[0]).index(max(sims[0]))]
    print(userQuery)
print(' '.join(userQuery))


1.0
['What', 'category', 'does', 'Keyboard', 'come', 'under?']
What category does Keyboard come under?
